In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Vertex Voice Agenda Agent — QLoRA Fine-Tuning (Kaggle)\n",
        "Entrena un adaptador LoRA sobre diálogos de llamadas telefónicas y tool-calling de agenda."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "!pip install -q unsloth trl transformers datasets accelerate bitsandbytes"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import json\n",
        "from pathlib import Path\n",
        "from datasets import Dataset\n",
        "from trl import SFTConfig, SFTTrainer\n",
        "from unsloth import FastLanguageModel\n",
        "\n",
        "BASE_MODEL = \"unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit\"\n",
        "DATA_PATH = Path(\"/kaggle/input/vertex-voice-agenda-data/train_calls_sft.jsonl\")\n",
        "if not DATA_PATH.exists():\n",
        "    DATA_PATH = Path(\"../data/train_calls_sft.jsonl\")\n",
        "\n",
        "rows = [json.loads(line) for line in DATA_PATH.read_text(encoding=\"utf-8\").splitlines() if line.strip()]\n",
        "dataset = Dataset.from_list(rows)\n",
        "\n",
        "model, tokenizer = FastLanguageModel.from_pretrained(\n",
        "    model_name=BASE_MODEL,\n",
        "    max_seq_length=2048,\n",
        "    load_in_4bit=True,\n",
        ")\n",
        "\n",
        "model = FastLanguageModel.get_peft_model(\n",
        "    model,\n",
        "    r=16,\n",
        "    target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"],\n",
        "    lora_alpha=16,\n",
        "    lora_dropout=0.0,\n",
        "    bias=\"none\",\n",
        "    use_gradient_checkpointing=\"unsloth\",\n",
        ")\n",
        "\n",
        "def format_chat(example):\n",
        "    text = tokenizer.apply_chat_template(example[\"messages\"], tokenize=False, add_generation_prompt=False)\n",
        "    return {\"text\": text}\n",
        "\n",
        "formatted_ds = dataset.map(format_chat)\n",
        "\n",
        "trainer = SFTTrainer(\n",
        "    model=model,\n",
        "    tokenizer=tokenizer,\n",
        "    train_dataset=formatted_ds,\n",
        "    args=SFTConfig(\n",
        "        output_dir=\"/kaggle/working/vertex-voice-agenda-lora\",\n",
        "        per_device_train_batch_size=2,\n",
        "        gradient_accumulation_steps=4,\n",
        "        num_train_epochs=3,\n",
        "        learning_rate=2e-4,\n",
        "        optim=\"adamw_torch_fused\",\n",
        "        dataset_text_field=\"text\",\n",
        "        logging_steps=1,\n",
        "        save_strategy=\"epoch\",\n",
        "        report_to=\"none\",\n",
        "    ),\n",
        ")\n",
        "\n",
        "trainer.train()\n",
        "model.save_pretrained(\"/kaggle/working/vertex-voice-agenda-lora-final\")\n",
        "tokenizer.save_pretrained(\"/kaggle/working/vertex-voice-agenda-lora-final\")"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}

import json
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": "Eres el agente de voz y agenda de Vertex Coders. Responde en JSON con assistant_reply y tool_calls.",
    },
    {
        "role": "user",
        "content": "Llama a Clínica Dental Coral Gables para agendar limpieza esta semana por la mañana.",
    },
    {
        "role": "assistant",
        "content": '{"assistant_reply":"Hola, llamo de parte de Denis Sánchez para consultar espacio esta semana por la mañana.","tool_calls":[{"tool":"check_calendar","arguments":{"start_iso":"2026-11-24T09:00:00-05:00","end_iso":"2026-11-24T12:00:00-05:00","calendar_label":"primary"}}]}',
    },
    {
        "role": "user",
        "content": "Contraparte (Clínica Dental Coral Gables): Sí, tenemos espacio mañana a las 10:00 AM. ¿Le confirmo ese horario?",
    },
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=220, temperature=0.2, do_sample=True)
decoded = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(decoded)